# San Juan Islands seascape data explorer

Run a bounded end-to-end workflow: configure a notebook-owned workspace, download real source data, build H3 r6/r8 marine support, derive bathymetry variables, and export one interactive HTML map.

Everything generated by this notebook stays under `notebooks/outputs/`. The 1:10m Natural Earth land mask is an **exploratory support mask**, not the toolkit's canonical territorial-water product or a legal boundary. GEBCO-derived values are not for navigation.

## 1. Configuration

The switches are intentionally on for true download, processing, and post-processing. Re-runs reuse cached downloads unless `FORCE_DOWNLOADS` is enabled.

In [28]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import FileLink, display


def find_toolkit_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "config/data/project.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the toolkit-seascape checkout.")


TOOLKIT_ROOT = find_toolkit_root(Path.cwd())
WORKSPACE_ROOT = TOOLKIT_ROOT / "notebooks/outputs"
PROJECT_CONFIG = Path("config/data/project.yaml")
SAN_JUAN_BBOX = (-123.35, 48.38, -122.70, 49.02)  # west, south, east, north
H3_RESOLUTIONS = (6, 8)
FINAL_HTML = WORKSPACE_ROOT / "outputs/san_juan_h3_bathymetry.html"

EXECUTE_DOWNLOADS = True
EXECUTE_PROCESSING = True
EXECUTE_POSTPROCESSING = True
FORCE_DOWNLOADS = False

notebook_root = TOOLKIT_ROOT / "notebooks"
source_root = str(TOOLKIT_ROOT / "src")
for path in (str(notebook_root), source_root):
    if path not in sys.path:
        sys.path.insert(0, path)

from _san_juan_workflow import prepare_workspace

PROJECT_CONFIG = prepare_workspace(
    TOOLKIT_ROOT,
    WORKSPACE_ROOT,
    SAN_JUAN_BBOX,
    H3_RESOLUTIONS,
    force_downloads=FORCE_DOWNLOADS,
)
print(f"Toolkit source: {TOOLKIT_ROOT}")
print(f"Notebook workspace: {WORKSPACE_ROOT}")
print(f"San Juan bbox: {SAN_JUAN_BBOX}")
print(f"Final interactive HTML: {FINAL_HTML}")

Toolkit source: /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape
Notebook workspace: /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs
San Juan bbox: (-123.35, 48.38, -122.7, 49.02)
Final interactive HTML: /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/outputs/san_juan_h3_bathymetry.html


### Validate path and bounding-box routing

In [29]:
from seascape.seafloor_physiography.bathymetry.pipeline import load_bathymetry_config

resolved_bathymetry = load_bathymetry_config(PROJECT_CONFIG)
resolved_bbox = tuple(resolved_bathymetry.bbox[key] for key in ("min_lon", "min_lat", "max_lon", "max_lat"))
assert resolved_bbox == SAN_JUAN_BBOX
assert resolved_bathymetry.raw_path.is_relative_to(WORKSPACE_ROOT)
assert resolved_bathymetry.processed_path.is_relative_to(WORKSPACE_ROOT)
display(pd.DataFrame([
    ("download", EXECUTE_DOWNLOADS),
    ("processing", EXECUTE_PROCESSING),
    ("post-processing", EXECUTE_POSTPROCESSING),
    ("bbox W,S,E,N", SAN_JUAN_BBOX),
    ("raw GEBCO", resolved_bathymetry.raw_path),
    ("processed bathymetry", resolved_bathymetry.processed_path),
    ("interactive HTML", FINAL_HTML),
], columns=["setting", "value"]))

,setting,value
0,download,True
1,processing,True
2,post-processing,True
3,"bbox W,S,E,N","(-123.35, 48.38, -122.7, 49.02)"
4,raw GEBCO,/Users/tylerstevenson/Documents/Code_Repos/Mar...
5,processed bathymetry,/Users/tylerstevenson/Documents/Code_Repos/Mar...
6,interactive HTML,/Users/tylerstevenson/Documents/Code_Repos/Mar...


## 2. True downloads

Download the bounded GEBCO GeoTIFF through GEBCO's queue API and the Natural Earth 1:10m land archive. Natural Earth land is subtracted from the bounding box to make the transparent exploratory water mask.

In [30]:
from _san_juan_workflow import run_downloads

water_mask_path = run_downloads(
    TOOLKIT_ROOT,
    WORKSPACE_ROOT,
    PROJECT_CONFIG,
    SAN_JUAN_BBOX,
    enabled=EXECUTE_DOWNLOADS,
    force_downloads=FORCE_DOWNLOADS,
)

Using cached download: /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/raw/natural_earth/ne_10m_land.zip
Exploratory water mask: /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/processed/domain/environmental_layer/seascape/spatial_support/water_geometry/TERRITORIAL_WATER_POLYGON.parquet
$ /Users/tylerstevenson/miniforge3/envs/orcacast/bin/python -m seascape --workspace /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs download bathymetry --config config/data/project.yaml


## 3. Processing

Build full and water-clipped H3 geometry, graph-shaped regional marine support at r6/r8, and all configured bathymetry summaries. Processing consumes the downloaded GEBCO raster without contacting the provider again.

In [31]:
from _san_juan_workflow import run_processing

run_processing(
    TOOLKIT_ROOT,
    WORKSPACE_ROOT,
    PROJECT_CONFIG,
    H3_RESOLUTIONS,
    enabled=EXECUTE_PROCESSING,
)

$ /Users/tylerstevenson/miniforge3/envs/orcacast/bin/python -m seascape.spatial_support.h3_geometry.build --config config/data/project.yaml --resolution 6 --resolution 8


2026-09-19 07:09:06,135 INFO __main__ Filling H3 resolution 6 with 5486 m buffer (base 2000 m + cell radius 3486 m).
2026-09-19 07:09:06,162 INFO __main__ Water area 6069635691 m², clipped hex area 6075227844 m², coverage ratio 100.1%
2026-09-19 07:09:06,168 INFO __main__ Filling H3 resolution 8 with 2498 m buffer (base 2000 m + cell radius 498 m).
2026-09-19 07:09:06,472 INFO __main__ Water area 6069635691 m², clipped hex area 6075227336 m², coverage ratio 100.1%


$ /Users/tylerstevenson/miniforge3/envs/orcacast/bin/python -m seascape.spatial_support.water_network.build --config config/data/project.yaml --resolution 6 --resolution 8 --overwrite


2026-09-19 07:09:07,452 INFO __main__ Building canonical H3 r8 marine support
2026-09-19 07:09:07,495 INFO __main__ Filled H3 r8 overlap support (4660 candidates) in 0.0s
2026-09-19 07:09:07,593 INFO __main__ Polygonized H3 r8 support in 0.1s
2026-09-19 07:09:07,734 INFO __main__ Clipped H3 r8 support to exact water geometry in 0.1s
2026-09-19 07:09:08,076 INFO __main__ Calculated H3 r8 geodesic areas/flags in 0.3s
2026-09-19 07:09:08,077 INFO __main__ Built H3 r8 geometry/support base (4660 wet cells) in 0.6s
2026-09-19 07:09:08,430 INFO seascape.spatial_support.water_network.graph Evaluated H3 r8 water edges: 13091
2026-09-19 07:09:08,431 INFO __main__ Evaluated H3 r8 neighbor passability (13091 candidates) in 0.3s
2026-09-19 07:09:08,454 INFO seascape.spatial_support.water_network.graph Unioned valid water edges in 0.0s
2026-09-19 07:09:08,461 INFO seascape.spatial_support.water_network.graph Derived deterministic water components in 0.0s
2026-09-19 07:09:08,465 INFO seascape.spatia

[
  "/Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/processed/domain/environmental_layer/seascape/spatial_support/h3_geometry/H3_MARINE_SUPPORT_RES_8.parquet",
  "/Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/processed/domain/environmental_layer/seascape/spatial_support/h3_geometry/H3_MODEL_AREA_SUPPORT_RES_8.parquet",
  "/Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/processed/domain/environmental_layer/seascape/spatial_support/h3_geometry/H3_MARINE_FULL_CELL_GEOMETRY_RES_8.parquet",
  "/Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/processed/domain/environmental_layer/seascape/spatial_support/h3_geometry/H3_MARINE_WATER_CLIPPED_GEOMETRY_RES_8.parquet",
  "/Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/proces

2026-09-19 07:09:10,972 INFO seascape.seafloor_physiography.bathymetry.build Using 4660 water H3 cells at resolution 8 from /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/data/processed/domain/environmental_layer/seascape/spatial_support/h3_geometry/H3_MODEL_AREA_SUPPORT_RES_8.parquet
2026-09-19 07:09:13,421 INFO seascape.seafloor_physiography.bathymetry.build Measured DISTANCE_TO_ISOBATH_6_1_M from 855 native-raster contour crossings
2026-09-19 07:09:13,423 INFO seascape.seafloor_physiography.bathymetry.build Measured DISTANCE_TO_ISOBATH_50_M from 2377 native-raster contour crossings
2026-09-19 07:09:13,424 INFO seascape.seafloor_physiography.bathymetry.build Measured DISTANCE_TO_ISOBATH_100_M from 1764 native-raster contour crossings
2026-09-19 07:09:13,425 INFO seascape.seafloor_physiography.bathymetry.build Measured DISTANCE_TO_ISOBATH_200_M from 906 native-raster contour crossings
2026-09-19 07:09:13,432 INFO seascape.seafloor_phy

## 4. Post-processing and interactive HTML

Render one self-contained Folium HTML with switchable H3 r6/r8 layers. Its selector exposes every numeric post-processed bathymetry variable: depth distribution, variability, local anomaly, depth-band composition, and isobath distances.

In [32]:
from _san_juan_workflow import run_postprocessing

run_postprocessing(
    TOOLKIT_ROOT,
    WORKSPACE_ROOT,
    PROJECT_CONFIG,
    FINAL_HTML,
    enabled=EXECUTE_POSTPROCESSING,
)

$ /Users/tylerstevenson/miniforge3/envs/orcacast/bin/python -m seascape --workspace /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs inspect bathymetry --config config/data/project.yaml --output /Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/outputs/san_juan_h3_bathymetry.html
/Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/outputs/san_juan_h3_bathymetry.html


## 5. Output and sanity checks

Verify identity, resolution, numeric domains, and artifact placement. Unknown raster support remains null; it is not converted to zero.

In [33]:
from _san_juan_workflow import validate_outputs

artifact_table, quality_table = validate_outputs(WORKSPACE_ROOT, FINAL_HTML)
display(artifact_table)
display(quality_table)
display(FileLink(FINAL_HTML, result_html_prefix="Open the interactive H3 map: "))

,artifact,path,exists
0,exploratory water mask,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
1,H3 r6 support,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
2,H3 r8 support,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
3,bathymetry r6,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
4,bathymetry r8,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
5,interactive HTML,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True


,H3 resolution,rows,numeric variables,cells with mean depth,minimum mean depth m,maximum mean depth m
0,6,134,28,127,1.764706,243.427184
1,8,4660,28,4384,1.000000,354.000000


/Users/tylerstevenson/Documents/Code_Repos/MarineCast/Toolkits/toolkit-seascape/notebooks/outputs/outputs/san_juan_h3_bathymetry.html

In [34]:
artifact_table

,artifact,path,exists
0,exploratory water mask,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
1,H3 r6 support,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
2,H3 r8 support,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
3,bathymetry r6,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
4,bathymetry r8,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True
5,interactive HTML,/Users/tylerstevenson/Documents/Code_Repos/Mar...,True


### Interpretation boundary

This validates live acquisition and bounded physical processing. The Natural Earth-derived mask is suitable for this explorer, not release promotion, navigation, legal boundary interpretation, or parity claims against the canonical multi-source territorial-water build. Source data, processed Parquet products, manifests, and the final HTML all remain under `notebooks/outputs/`.